# Notebook 3 – Fire Incident Dashboard Deliverables
**Inputs:** `incidents_clean.csv` (Notebook 1) and `fire_ml_models/` outputs (Notebook 2 – ML)

| Deliverable | Section |
|---|---|
| Executive KPI dashboard | 3 |
| Trend charts | 4 |
| Heat maps | 5 |
| Response performance scorecards | 6 |
| Cause analysis dashboard | 7 |
| Equipment risk rankings | 8 |
| ML high-risk area forecast | 9 |
| Downloadable reports (HTML dashboard, Excel report, PNG charts) | 10 |
| Interactive **Streamlit** app (`fire_dashboard_app.py`) | 11 |

## 1. Setup

In [1]:
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)

DATA_DIR = Path(".")
ML_DIR = DATA_DIR / "fire_ml_models"
OUT_DIR = Path("fire_dashboard_output"); OUT_DIR.mkdir(exist_ok=True)
(OUT_DIR / "charts").mkdir(exist_ok=True)

RESPONSE_TARGET_MIN = 3     # KPI target: minutes from call to fire-crew response
NATURE_COLORS = {"EQUIPMENT": "#C0392B", "BUILDING": "#E67E22", "3RD PARTY": "#2E86C1", "BUSH/SCRAP": "#27AE60"}
SEV_COLORS = {"High": "#C0392B", "Medium": "#F39C12", "Low": "#27AE60"}
MONTH_ORDER = ["January", "February", "March", "April", "May", "June", "July", "August",
               "September", "October", "November", "December"]
TEMPLATE = "plotly_white"
FIGS = {}   # every figure is stored here for the HTML / PNG export

## 2. Load data

In [2]:
inc = pd.read_csv(DATA_DIR / "incidents_clean.csv", parse_dates=["incident_date", "call_datetime"])
inc.columns = inc.columns.str.replace("\ufeff", "")
inc["month_name"] = pd.Categorical(inc["month_name"], MONTH_ORDER, ordered=True)
inc["ym"] = inc["incident_date"].dt.to_period("M").astype(str)

def read_ml(name):
    p = ML_DIR / name
    if not p.exists():
        print("⚠ not found (run ML notebook):", p); return None
    d = pd.read_csv(p); d.columns = d.columns.str.replace("\ufeff", ""); return d

area_risk = read_ml("area_risk_30d.csv")
metrics = read_ml("model_metrics.csv")
inc_pred = read_ml("incident_predictions.csv")
REF_DATE = inc["incident_date"].max()
print(f"{len(inc)} incidents | {inc.incident_date.min():%d-%b-%Y} → {REF_DATE:%d-%b-%Y}")

61 incidents | 01-Jan-2026 → 02-Oct-2026


## 3. Executive KPI dashboard

In [3]:
months = sorted(inc["ym"].unique())
last_m, prev_m = months[-1], (months[-2] if len(months) > 1 else None)
resp = inc["response_minutes"].dropna()
kpi = {
    "Total fires (YTD)": len(inc),
    f"Fires in {last_m}": int((inc.ym == last_m).sum()),
    "Avg monthly fires": round(inc.groupby("ym").size().mean(), 1),
    "Avg response (min)": round(resp.mean(), 2),
    f"% response ≤ {RESPONSE_TARGET_MIN} min": round((resp <= RESPONSE_TARGET_MIN).mean() * 100, 1),
    "% out before fire crew": round(inc["out_before_fire_crew"].mean() * 100, 1),
    "Injury incidents": int(inc["injury_reported"].sum()),
    "High-severity fires": int((inc["severity_level"] == "High").sum()),
    "Repeat-equipment fires": int(inc["is_repeat_equipment"].sum()),
    "Top area": f"{inc.area.value_counts().idxmax()} ({inc.area.value_counts().max()})",
    "Top cause": f"{inc.cause_category.value_counts().idxmax()} ({inc.cause_category.value_counts().max()})",
    "Top equipment": f"{inc.primary_tag_base.value_counts().idxmax()} ({inc.primary_tag_base.value_counts().max()})",
}
kpi_df = pd.DataFrame(kpi.items(), columns=["KPI", "Value"])

num_kpis = [(k, v) for k, v in kpi.items() if isinstance(v, (int, float))][:8]
fig = make_subplots(rows=2, cols=4, specs=[[{"type": "indicator"}] * 4] * 2)
for i, (k, v) in enumerate(num_kpis):
    delta = None
    if k.startswith("Fires in") and prev_m:
        delta = {"reference": int((inc.ym == prev_m).sum()), "increasing": {"color": "#C0392B"},
                 "decreasing": {"color": "#27AE60"}}
    fig.add_trace(go.Indicator(mode="number+delta" if delta else "number", value=v, delta=delta,
                               title={"text": k, "font": {"size": 14}}, number={"font": {"size": 38, "color": "#7B241C"}}),
                  row=i // 4 + 1, col=i % 4 + 1)
fig.update_layout(height=330, title="Executive KPI dashboard", template=TEMPLATE, margin=dict(t=60, b=10))
FIGS["01_kpi_cards"] = fig; fig.show()

fig = make_subplots(rows=1, cols=3, specs=[[{"type": "domain"}] * 3],
                    subplot_titles=["By nature", "By severity", "By zone"])
for j, col in enumerate(["fire_nature", "severity_level", "zone"], 1):
    vc = inc[col].value_counts()
    cmap = NATURE_COLORS if col == "fire_nature" else SEV_COLORS if col == "severity_level" else None
    fig.add_trace(go.Pie(labels=vc.index, values=vc.values, hole=0.55,
                         marker_colors=[cmap.get(x) for x in vc.index] if cmap else None), 1, j)
fig.update_layout(height=380, template=TEMPLATE, title="Incident mix")
FIGS["02_mix"] = fig; fig.show()
kpi_df

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

## 4. Trend charts

In [ ]:
m = inc.groupby(["ym", "fire_nature"]).size().reset_index(name="fires")
fig = px.bar(m, x="ym", y="fires", color="fire_nature", color_discrete_map=NATURE_COLORS, text_auto=True,
             title="Monthly fires by nature", labels={"ym": "Month"}, template=TEMPLATE)
FIGS["03_monthly_nature"] = fig; fig.show()

t = inc.groupby("ym").size().reset_index(name="fires")
t["3-month avg"] = t["fires"].rolling(3, min_periods=1).mean()
t["cumulative"] = t["fires"].cumsum()
fig = go.Figure()
fig.add_bar(x=t["ym"], y=t["fires"], name="Fires", marker_color="#E6B0AA")
fig.add_scatter(x=t["ym"], y=t["3-month avg"], name="3-month avg", line=dict(color="#C0392B", width=3))
fig.add_scatter(x=t["ym"], y=t["cumulative"], name="Cumulative", yaxis="y2", line=dict(color="#2C3E50", dash="dot"))
fig.update_layout(title="Fire trend – monthly, rolling average & cumulative", template=TEMPLATE,
                  yaxis2=dict(overlaying="y", side="right", title="Cumulative"))
FIGS["04_trend"] = fig; fig.show()

s = inc.groupby(["ym", "severity_level"]).size().reset_index(name="fires")
fig = px.line(s, x="ym", y="fires", color="severity_level", markers=True, color_discrete_map=SEV_COLORS,
              title="Severity trend", template=TEMPLATE)
FIGS["05_severity_trend"] = fig; fig.show()

w = inc.set_index("incident_date").resample("W")["incident_id"].count().reset_index(name="fires")
fig = px.area(w, x="incident_date", y="fires", title="Weekly fire count", template=TEMPLATE)
fig.update_traces(line_color="#C0392B")
FIGS["06_weekly"] = fig; fig.show()

## 5. Heat maps

In [ ]:
hm = inc.pivot_table(index="area", columns="month_name", values="incident_id", aggfunc="count", fill_value=0, observed=True)
hm = hm.loc[hm.sum(axis=1).sort_values(ascending=False).index]
fig = px.imshow(hm, text_auto=True, color_continuous_scale="Reds", aspect="auto", title="Heat map – Area × Month",
                labels=dict(x="", y="", color="Fires"), template=TEMPLATE)
fig.update_layout(height=600); FIGS["07_heat_area_month"] = fig; fig.show()

days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
hh = (inc.dropna(subset=["hour"]).pivot_table(index="day_name", columns="hour", values="incident_id",
                                              aggfunc="count", fill_value=0)
      .reindex(days).reindex(columns=range(24)).fillna(0).astype(int))
fig = px.imshow(hh, text_auto=True, color_continuous_scale="OrRd", aspect="auto",
                title="Heat map – Day of week × Hour of call", labels=dict(x="Hour", y="", color="Fires"), template=TEMPLATE)
FIGS["08_heat_day_hour"] = fig; fig.show()

zc = inc.pivot_table(index="zone", columns="cause_category", values="incident_id", aggfunc="count", fill_value=0)
fig = px.imshow(zc, text_auto=True, color_continuous_scale="YlOrRd", aspect="auto",
                title="Heat map – Zone × Cause", labels=dict(x="", y="", color="Fires"), template=TEMPLATE)
fig.update_layout(height=500); FIGS["09_heat_zone_cause"] = fig; fig.show()

## 6. Response performance scorecards
Target: response within `RESPONSE_TARGET_MIN` minutes. RAG: 🟢 ≥95 % within target, 🟠 ≥80 %, 🔴 below.

In [ ]:
r = inc.dropna(subset=["response_minutes"])
def card(g):
    return pd.Series({"incidents": len(g), "avg_min": round(g.response_minutes.mean(), 2),
                      "median_min": g.response_minutes.median(), "max_min": g.response_minutes.max(),
                      "pct_within_target": round((g.response_minutes <= RESPONSE_TARGET_MIN).mean() * 100, 1),
                      "pct_out_before_crew": round(g.out_before_fire_crew.mean() * 100, 1),
                      "pct_fire_crew_extinguished": round((g.extinguished_by == "Fire crew").mean() * 100, 1)})
def rag(p):
    return "GREEN" if p >= 95 else "AMBER" if p >= 80 else "RED"

score_month = r.groupby("ym").apply(card).reset_index().rename(columns={"ym": "month"})
score_zone = r.groupby("zone").apply(card).reset_index().sort_values("avg_min", ascending=False)
score_shift = r.groupby("shift").apply(card).reset_index()
for d in (score_month, score_zone, score_shift):
    d["status"] = d["pct_within_target"].apply(rag)

fig = make_subplots(rows=1, cols=2, subplot_titles=["Average response by month (min)", "% within target by zone"])
fig.add_bar(x=score_month["month"], y=score_month["avg_min"], marker_color="#2E86C1", name="Avg min", row=1, col=1)
fig.add_hline(y=RESPONSE_TARGET_MIN, line_dash="dash", line_color="red", row=1, col=1)
cols = score_zone["status"].map({"GREEN": "#27AE60", "AMBER": "#F39C12", "RED": "#C0392B"})
fig.add_bar(x=score_zone["pct_within_target"], y=score_zone["zone"], orientation="h", marker_color=cols,
            name="% within target", row=1, col=2)
fig.update_layout(height=420, template=TEMPLATE, showlegend=False, title="Response performance scorecard")
FIGS["10_response_scorecard"] = fig; fig.show()

eb = inc["extinguished_by"].value_counts().reset_index(); eb.columns = ["extinguished_by", "fires"]
fig = px.bar(eb, x="fires", y="extinguished_by", orientation="h", text_auto=True, title="Who extinguished the fire",
             template=TEMPLATE, color_discrete_sequence=["#7B241C"])
FIGS["11_extinguished_by"] = fig; fig.show()

def rag_style(v):
    return {"GREEN": "background-color:#D5F5E3", "AMBER": "background-color:#FDEBD0",
            "RED": "background-color:#FADBD8"}.get(v, "")
_m = "map" if hasattr(inc.style, "map") else "applymap"   # pandas <2.1 compatibility
display(getattr(score_month.style, _m)(rag_style, subset=["status"]))
display(getattr(score_zone.style, _m)(rag_style, subset=["status"]))

## 7. Cause analysis dashboard

In [ ]:
p = inc["cause_category"].value_counts().reset_index(); p.columns = ["cause", "fires"]
p["cum_pct"] = (p["fires"].cumsum() / p["fires"].sum() * 100).round(1)
fig = go.Figure()
fig.add_bar(x=p["cause"], y=p["fires"], name="Fires", marker_color="#C0392B", text=p["fires"])
fig.add_scatter(x=p["cause"], y=p["cum_pct"], name="Cumulative %", yaxis="y2", mode="lines+markers", line_color="#2C3E50")
fig.add_hline(y=80, line_dash="dot", line_color="grey", yref="y2")
fig.update_layout(title="Pareto of fire causes (80/20)", template=TEMPLATE,
                  yaxis2=dict(overlaying="y", side="right", range=[0, 105], ticksuffix="%"))
FIGS["12_cause_pareto"] = fig; fig.show()

fig = px.sunburst(inc, path=["zone", "area", "cause_category"], title="Zone → Area → Cause", template=TEMPLATE,
                  color="zone")
fig.update_layout(height=600); FIGS["13_cause_sunburst"] = fig; fig.show()

cf = inc.pivot_table(index="cause_category", columns="fuel_type", values="incident_id", aggfunc="count", fill_value=0)
fig = px.imshow(cf, text_auto=True, color_continuous_scale="Reds", aspect="auto", title="Cause × Material burned",
                template=TEMPLATE)
FIGS["14_cause_fuel"] = fig; fig.show()

oa = inc.groupby(["operating_activity", "severity_level"]).size().reset_index(name="fires")
fig = px.bar(oa, x="operating_activity", y="fires", color="severity_level", color_discrete_map=SEV_COLORS,
             title="Plant activity at time of fire (by severity)", template=TEMPLATE)
FIGS["15_activity"] = fig; fig.show()

cause_table = (inc.groupby("cause_category")
               .agg(fires=("incident_id", "count"), high_severity=("severity_level", lambda s: (s == "High").sum()),
                    injuries=("injury_reported", "sum"), areas=("area", lambda s: ", ".join(s.value_counts().index[:3])),
                    last_fire=("incident_date", "max"))
               .sort_values("fires", ascending=False).reset_index())
cause_table

## 8. Equipment risk rankings
Risk score = fires × 10 + avg severity (1-3) × 10 + injuries × 15 + recency (half-life 90 days) × 20

In [ ]:
def equipment_risk(df, ref_date):
    e = df.dropna(subset=["primary_tag_base"]).copy()
    e["sev_pts"] = e["severity_level"].map({"Low": 1, "Medium": 2, "High": 3}).fillna(1)
    g = e.groupby("primary_tag_base").agg(
        area=("area", "first"), equipment_type=("equipment_type", "first"),
        fires=("incident_id", "count"), avg_severity=("sev_pts", "mean"),
        injuries=("injury_reported", "sum"), first_fire=("incident_date", "min"),
        last_fire=("incident_date", "max"), causes=("cause_category", lambda s: ", ".join(s.unique()))).reset_index()
    g["days_since_last"] = (ref_date - g["last_fire"]).dt.days
    g["recency_factor"] = (0.5 ** (g["days_since_last"] / 90)).round(3)
    g["risk_score"] = (g["fires"] * 10 + g["avg_severity"] * 10 + g["injuries"] * 15 + g["recency_factor"] * 20).round(1)
    g["risk_rank"] = g["risk_score"].rank(ascending=False, method="min").astype(int)
    g["risk_band"] = pd.cut(g["risk_score"], [-1, 30, 50, 1e9], labels=["Low", "Medium", "High"]).astype(str)
    return g.sort_values("risk_rank").reset_index(drop=True)

equip_risk = equipment_risk(inc, REF_DATE)
top = equip_risk.head(15).iloc[::-1]
fig = px.bar(top, x="risk_score", y="primary_tag_base", color="risk_band", orientation="h",
             color_discrete_map=SEV_COLORS, text="fires", hover_data=["area", "equipment_type", "last_fire", "causes"],
             title="Equipment risk ranking – Top 15 (bar label = number of fires)", template=TEMPLATE)
fig.update_layout(height=550); FIGS["16_equipment_risk"] = fig; fig.show()

et = inc.groupby("equipment_type").agg(fires=("incident_id", "count"),
                                       high_severity=("severity_level", lambda s: (s == "High").sum())).reset_index()
fig = px.treemap(et, path=["equipment_type"], values="fires", color="high_severity", color_continuous_scale="Reds",
                 title="Fires by equipment type (colour = high-severity count)", template=TEMPLATE)
FIGS["17_equipment_treemap"] = fig; fig.show()

rep = inc.dropna(subset=["primary_tag_base"])
rep = rep[rep["primary_tag_base"].isin(equip_risk.loc[equip_risk.fires >= 2, "primary_tag_base"])]
fig = px.scatter(rep, x="incident_date", y="primary_tag_base", color="severity_level", color_discrete_map=SEV_COLORS,
                 size=np.full(len(rep), 12), title="Repeat-failure timeline (equipment with ≥2 fires)", template=TEMPLATE)
FIGS["18_repeat_timeline"] = fig; fig.show()
equip_risk.head(15)

## 9. ML forecast – high-risk areas (from the ML notebook)

In [ ]:
if area_risk is not None:
    ar = area_risk.sort_values("risk_score_30d")
    fig = px.bar(ar, x="risk_score_30d", y="area", color="risk_band", orientation="h", color_discrete_map=SEV_COLORS,
                 hover_data=["prob_ml_30d", "prob_hist_30d", "fires_ytd", "fires_last_90d"],
                 title="Predicted probability of ≥1 fire in the next 30 days", template=TEMPLATE)
    fig.update_xaxes(tickformat=".0%"); fig.update_layout(height=550)
    FIGS["19_ml_area_risk"] = fig; fig.show()
if metrics is not None:
    display(metrics)
if inc_pred is not None:
    pc = [c for c in inc_pred.columns if c.startswith("pred_category_")][0]
    acc = (inc_pred["actual_category_model"] == inc_pred[pc]).mean()
    print(f"Category model – out-of-fold accuracy: {acc:.0%}")

## 10. EXPORT – downloadable reports
* `Fire_Dashboard_<date>.html` – full interactive dashboard (open in any browser, share by email)
* `Fire_Dashboard_Report_<date>.xlsx` – KPI summary, scorecards, rankings, data + chart images
* `charts/*.png` – static images for PowerPoint

In [ ]:
from openpyxl import load_workbook
from openpyxl.drawing.image import Image as XLImage
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

def export_dashboard(out_dir=OUT_DIR):
    out_dir = Path(out_dir); stamp = datetime.now().strftime("%Y%m%d_%H%M")

    # 1) Interactive HTML dashboard
    html_path = out_dir / f"Fire_Dashboard_{stamp}.html"
    kpi_html = kpi_df.to_html(index=False, classes="kpi", border=0)
    parts = [f"<html><head><meta charset='utf-8'><title>Fire Dashboard</title>"
             "<style>body{font-family:Arial;margin:24px;background:#fafafa}h1{color:#7B241C}"
             ".kpi td,.kpi th{padding:6px 14px;border-bottom:1px solid #ddd}.card{background:#fff;margin:14px 0;"
             "padding:8px;border-radius:8px;box-shadow:0 1px 4px #ccc}</style></head><body>",
             f"<h1>Fire Emergency Analytics Dashboard</h1><p>Generated {datetime.now():%d %b %Y %H:%M} | "
             f"{len(inc)} incidents | {inc.incident_date.min():%d %b %Y} – {REF_DATE:%d %b %Y}</p>",
             f"<div class='card'><h2>KPI summary</h2>{kpi_html}</div>"]
    for i, (name, fig) in enumerate(FIGS.items()):
        parts.append("<div class='card'>" + fig.to_html(full_html=False, include_plotlyjs="cdn" if i == 0 else False) + "</div>")
    parts.append("</body></html>")
    html_path.write_text("\n".join(parts), encoding="utf-8")

    # 2) PNG charts (needs `pip install kaleido`; skipped gracefully if unavailable)
    pngs = {}
    try:
        for name, fig in FIGS.items():
            p = out_dir / "charts" / f"{name}.png"
            fig.write_image(p, width=1200, height=fig.layout.height or 500, scale=1)
            pngs[name] = p
    except Exception as e:
        print("PNG export skipped (install kaleido):", str(e)[:120])

    # 3) Excel report
    xlsx = out_dir / f"Fire_Dashboard_Report_{stamp}.xlsx"
    monthly = inc.pivot_table(index="ym", columns="fire_nature", values="incident_id", aggfunc="count", fill_value=0)
    monthly["TOTAL"] = monthly.sum(axis=1)
    sheets = {
        "KPI_Summary": kpi_df,
        "Monthly_Trend": monthly.reset_index().rename(columns={"ym": "month"}),
        "Area_x_Month": hm.reset_index(),
        "Response_by_Month": score_month,
        "Response_by_Zone": score_zone,
        "Response_by_Shift": score_shift,
        "Cause_Analysis": cause_table,
        "Equipment_Risk": equip_risk,
        "Incidents": inc.drop(columns=["ym"]),
    }
    if area_risk is not None:
        sheets["ML_Area_Risk_30d"] = area_risk
    with pd.ExcelWriter(xlsx, engine="openpyxl") as xw:
        for n, d in sheets.items():
            d.to_excel(xw, sheet_name=n[:31], index=False)
    wb = load_workbook(xlsx)
    rag_fill = {"GREEN": "D5F5E3", "AMBER": "FDEBD0", "RED": "FADBD8", "High": "FADBD8", "Medium": "FDEBD0", "Low": "D5F5E3"}
    for ws in wb.worksheets:
        for c in ws[1]:
            c.font = Font(name="Arial", bold=True, color="FFFFFF"); c.fill = PatternFill("solid", start_color="7B241C")
            c.alignment = Alignment(wrap_text=True, vertical="center")
        for row in ws.iter_rows(min_row=2):
            for c in row:
                c.font = Font(name="Arial", size=10)
                if isinstance(c.value, str) and c.value in rag_fill:
                    c.fill = PatternFill("solid", start_color=rag_fill[c.value])
                if c.is_date:
                    c.number_format = "dd-mmm-yyyy"
        ws.freeze_panes = "A2"; ws.auto_filter.ref = ws.dimensions
        for i, col in enumerate(ws.iter_cols(min_row=1, max_row=min(ws.max_row, 100)), 1):
            wdt = max(len(str(c.value)) if c.value is not None else 0 for c in col)
            ws.column_dimensions[get_column_letter(i)].width = min(max(10, wdt + 2), 50)
    if pngs:
        ws = wb.create_sheet("Charts", 1); r = 1
        for name, p in pngs.items():
            img = XLImage(str(p)); img.width, img.height = 900, int(900 * img.height / img.width)
            ws.add_image(img, f"A{r}"); r += int(img.height / 20) + 2
    wb.save(xlsx)

    print("Exported:")
    print("  ", html_path.resolve()); print("  ", xlsx.resolve())
    print("  ", len(pngs), "PNG charts in", (out_dir / "charts").resolve())
    return html_path, xlsx

html_file, xlsx_file = export_dashboard()

## 11. Streamlit interactive dashboard
The cell below writes `fire_dashboard_app.py` next to the CSV files. Then run in a terminal / Anaconda prompt:
```
streamlit run fire_dashboard_app.py
```
Features: sidebar filters (date, zone, nature, severity), 8 tabs (KPIs, trends, heat maps, response scorecard, cause analysis,
equipment risk, ML forecast, reports) and **download buttons** for Excel / CSV reports.

In [ ]:
APP_CODE = r"""'''
Fire Incident Analytics Dashboard (Streamlit)
Run:  streamlit run fire_dashboard_app.py
Needs: incidents_clean.csv  (+ optional fire_ml_models/ outputs from the ML notebook)
'''
import io
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import streamlit as st

st.set_page_config(page_title="Fire Incident Dashboard", page_icon="🔥", layout="wide")

DATA_DIR = Path(__file__).parent
ML_DIR = DATA_DIR / "fire_ml_models"
RESPONSE_TARGET_MIN = 3          # target minutes from call to fire-crew response
NATURE_COLORS = {"EQUIPMENT": "#C0392B", "BUILDING": "#E67E22", "3RD PARTY": "#2E86C1", "BUSH/SCRAP": "#27AE60"}
SEV_COLORS = {"High": "#C0392B", "Medium": "#F39C12", "Low": "#27AE60"}
MONTH_ORDER = ["January", "February", "March", "April", "May", "June", "July", "August",
               "September", "October", "November", "December"]


@st.cache_data
def load():
    inc = pd.read_csv(DATA_DIR / "incidents_clean.csv", parse_dates=["incident_date", "call_datetime"])
    inc.columns = inc.columns.str.replace("\ufeff", "")
    inc["month_name"] = pd.Categorical(inc["month_name"], MONTH_ORDER, ordered=True)
    inc["ym"] = inc["incident_date"].dt.to_period("M").astype(str)
    ml = {}
    for k, f in {"area_risk": "area_risk_30d.csv", "metrics": "model_metrics.csv",
                 "inc_pred": "incident_predictions.csv"}.items():
        p = ML_DIR / f
        if p.exists():
            d = pd.read_csv(p); d.columns = d.columns.str.replace("\ufeff", ""); ml[k] = d
    return inc, ml


def equipment_risk(df, ref_date):
    e = df.dropna(subset=["primary_tag_base"]).copy()
    if e.empty:
        return pd.DataFrame()
    sev = {"Low": 1, "Medium": 2, "High": 3}
    e["sev_pts"] = e["severity_level"].map(sev).fillna(1)
    g = e.groupby("primary_tag_base").agg(
        area=("area", "first"), equipment_type=("equipment_type", "first"),
        fires=("incident_id", "count"), avg_severity=("sev_pts", "mean"),
        injuries=("injury_reported", "sum"), last_fire=("incident_date", "max"),
        first_fire=("incident_date", "min")).reset_index()
    g["days_since_last"] = (ref_date - g["last_fire"]).dt.days
    g["recency_factor"] = 0.5 ** (g["days_since_last"] / 90)
    g["risk_score"] = (g["fires"] * 10 + g["avg_severity"] * 10 + g["injuries"] * 15 + g["recency_factor"] * 20).round(1)
    g["risk_rank"] = g["risk_score"].rank(ascending=False, method="min").astype(int)
    g["risk_band"] = pd.cut(g["risk_score"], [-1, 30, 50, 1e9], labels=["Low", "Medium", "High"])
    return g.sort_values("risk_rank")


def to_excel(sheets):
    buf = io.BytesIO()
    with pd.ExcelWriter(buf, engine="openpyxl") as xw:
        for n, d in sheets.items():
            d.to_excel(xw, sheet_name=n[:31], index=False)
    return buf.getvalue()


inc, ml = load()

# ---------------- Sidebar filters ----------------
st.sidebar.title("🔥 Filters")
dmin, dmax = inc["incident_date"].min().date(), inc["incident_date"].max().date()
dr = st.sidebar.date_input("Date range", (dmin, dmax), min_value=dmin, max_value=dmax)
zones = st.sidebar.multiselect("Zone", sorted(inc["zone"].unique()), default=sorted(inc["zone"].unique()))
natures = st.sidebar.multiselect("Fire nature", sorted(inc["fire_nature"].unique()), default=sorted(inc["fire_nature"].unique()))
sevs = st.sidebar.multiselect("Severity", ["High", "Medium", "Low"], default=["High", "Medium", "Low"])
start, end = (dr if isinstance(dr, (list, tuple)) and len(dr) == 2 else (dmin, dmax))
f = inc[(inc["incident_date"].dt.date >= start) & (inc["incident_date"].dt.date <= end)
        & inc["zone"].isin(zones) & inc["fire_nature"].isin(natures) & inc["severity_level"].isin(sevs)]
if f.empty:
    st.warning("No incidents for the selected filters."); st.stop()

st.title("Fire Emergency Analytics Dashboard")
st.caption(f"{len(f)} incidents | {start:%d %b %Y} – {end:%d %b %Y} | response target ≤ {RESPONSE_TARGET_MIN} min")

tabs = st.tabs(["📊 Executive KPIs", "📈 Trends", "🗺️ Heat maps", "⏱️ Response scorecard",
                "🔍 Cause analysis", "⚙️ Equipment risk", "🤖 ML risk forecast", "📥 Reports"])

# ---------------- Executive KPIs ----------------
with tabs[0]:
    last_m = f["ym"].max()
    months = sorted(f["ym"].unique())
    prev_m = months[-2] if len(months) > 1 else None
    resp = f["response_minutes"].dropna()
    c = st.columns(4)
    c[0].metric("Total fires", len(f),
                f"{(f.ym == last_m).sum() - (f.ym == prev_m).sum():+d} vs prev month" if prev_m else None, delta_color="inverse")
    c[1].metric("Avg response (min)", f"{resp.mean():.1f}", f"{(resp <= RESPONSE_TARGET_MIN).mean():.0%} within target")
    c[2].metric("Out before fire crew", f"{f['out_before_fire_crew'].mean():.0%}")
    c[3].metric("Injury incidents", int(f["injury_reported"].sum()))
    c = st.columns(4)
    c[0].metric("High-severity fires", int((f["severity_level"] == "High").sum()))
    c[1].metric("Repeat-equipment fires", int(f["is_repeat_equipment"].sum()))
    c[2].metric("Top area", f["area"].value_counts().idxmax(), f"{f['area'].value_counts().max()} fires", delta_color="off")
    c[3].metric("Top cause", f["cause_category"].value_counts().idxmax(), delta_color="off")
    a, b = st.columns(2)
    a.plotly_chart(px.pie(f, names="fire_nature", hole=0.55, color="fire_nature", color_discrete_map=NATURE_COLORS,
                          title="Fires by nature"), use_container_width=True)
    b.plotly_chart(px.pie(f, names="severity_level", hole=0.55, color="severity_level", color_discrete_map=SEV_COLORS,
                          title="Fires by severity"), use_container_width=True)

# ---------------- Trends ----------------
with tabs[1]:
    m = f.groupby(["ym", "fire_nature"]).size().reset_index(name="fires")
    st.plotly_chart(px.bar(m, x="ym", y="fires", color="fire_nature", color_discrete_map=NATURE_COLORS,
                           title="Monthly fires by nature", labels={"ym": "Month"}), use_container_width=True)
    t = f.groupby("ym").size().reset_index(name="fires")
    t["cumulative"] = t["fires"].cumsum(); t["3-mo avg"] = t["fires"].rolling(3, min_periods=1).mean()
    fig = go.Figure()
    fig.add_bar(x=t["ym"], y=t["fires"], name="Fires", marker_color="#E6B0AA")
    fig.add_scatter(x=t["ym"], y=t["3-mo avg"], name="3-month avg", line=dict(color="#C0392B", width=3))
    fig.add_scatter(x=t["ym"], y=t["cumulative"], name="Cumulative", yaxis="y2", line=dict(dash="dot", color="#2C3E50"))
    fig.update_layout(title="Trend, rolling average & cumulative", yaxis2=dict(overlaying="y", side="right", title="Cumulative"))
    st.plotly_chart(fig, use_container_width=True)
    s = f.groupby(["ym", "severity_level"]).size().reset_index(name="fires")
    st.plotly_chart(px.line(s, x="ym", y="fires", color="severity_level", markers=True, color_discrete_map=SEV_COLORS,
                            title="Severity trend"), use_container_width=True)

# ---------------- Heat maps ----------------
with tabs[2]:
    hm = f.pivot_table(index="area", columns="month_name", values="incident_id", aggfunc="count", fill_value=0, observed=True)
    hm = hm.loc[hm.sum(axis=1).sort_values(ascending=False).index]
    st.plotly_chart(px.imshow(hm, text_auto=True, color_continuous_scale="Reds", aspect="auto",
                              title="Area × Month"), use_container_width=True)
    days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
    hh = f.dropna(subset=["hour"]).pivot_table(index="day_name", columns="hour", values="incident_id",
                                               aggfunc="count", fill_value=0).reindex(days).fillna(0)
    hh = hh.reindex(columns=range(24), fill_value=0)
    st.plotly_chart(px.imshow(hh, text_auto=True, color_continuous_scale="OrRd", aspect="auto",
                              title="Day of week × Hour of call", labels=dict(x="Hour", y="")), use_container_width=True)
    zc = f.pivot_table(index="zone", columns="cause_category", values="incident_id", aggfunc="count", fill_value=0)
    st.plotly_chart(px.imshow(zc, text_auto=True, color_continuous_scale="YlOrRd", aspect="auto",
                              title="Zone × Cause"), use_container_width=True)

# ---------------- Response scorecard ----------------
with tabs[3]:
    r = f.dropna(subset=["response_minutes"])
    def card(g):
        return pd.Series({"incidents": len(g), "avg_min": g["response_minutes"].mean(),
                          "median_min": g["response_minutes"].median(), "max_min": g["response_minutes"].max(),
                          "pct_within_target": (g["response_minutes"] <= RESPONSE_TARGET_MIN).mean(),
                          "pct_out_before_crew": g["out_before_fire_crew"].mean()})
    by_m = r.groupby("ym").apply(card).reset_index()
    by_z = r.groupby("zone").apply(card).reset_index().sort_values("avg_min", ascending=False)
    by_m["status"] = np.where(by_m["pct_within_target"] >= 0.95, "🟢 On target",
                              np.where(by_m["pct_within_target"] >= 0.8, "🟠 Watch", "🔴 Below target"))
    c = st.columns(3)
    c[0].metric("Average response", f"{r.response_minutes.mean():.2f} min")
    c[1].metric("Within target", f"{(r.response_minutes <= RESPONSE_TARGET_MIN).mean():.0%}")
    c[2].metric("Slowest", f"{r.response_minutes.max():.0f} min")
    st.subheader("Monthly scorecard")
    st.dataframe(by_m.style.format({"avg_min": "{:.2f}", "median_min": "{:.1f}", "max_min": "{:.0f}",
                                    "pct_within_target": "{:.0%}", "pct_out_before_crew": "{:.0%}", "incidents": "{:.0f}"}),
                 use_container_width=True)
    st.subheader("Zone scorecard")
    st.dataframe(by_z.style.format({"avg_min": "{:.2f}", "median_min": "{:.1f}", "max_min": "{:.0f}",
                                    "pct_within_target": "{:.0%}", "pct_out_before_crew": "{:.0%}", "incidents": "{:.0f}"}),
                 use_container_width=True)
    eb = f["extinguished_by"].value_counts().reset_index(); eb.columns = ["extinguished_by", "fires"]
    st.plotly_chart(px.bar(eb, x="fires", y="extinguished_by", orientation="h", title="Who put the fire out"),
                    use_container_width=True)

# ---------------- Cause analysis ----------------
with tabs[4]:
    p = f["cause_category"].value_counts().reset_index(); p.columns = ["cause", "fires"]
    p["cum_pct"] = p["fires"].cumsum() / p["fires"].sum() * 100
    fig = go.Figure()
    fig.add_bar(x=p["cause"], y=p["fires"], name="Fires", marker_color="#C0392B")
    fig.add_scatter(x=p["cause"], y=p["cum_pct"], name="Cumulative %", yaxis="y2", mode="lines+markers")
    fig.update_layout(title="Pareto of causes", yaxis2=dict(overlaying="y", side="right", range=[0, 105], ticksuffix="%"))
    st.plotly_chart(fig, use_container_width=True)
    a, b = st.columns(2)
    a.plotly_chart(px.sunburst(f, path=["zone", "area", "cause_category"], title="Zone → Area → Cause"),
                   use_container_width=True)
    fu = f["fuel_type"].value_counts().reset_index(); fu.columns = ["fuel", "fires"]
    b.plotly_chart(px.bar(fu, x="fires", y="fuel", orientation="h", title="Material burned"), use_container_width=True)
    oa = f.groupby(["operating_activity", "severity_level"]).size().reset_index(name="fires")
    st.plotly_chart(px.bar(oa, x="operating_activity", y="fires", color="severity_level", color_discrete_map=SEV_COLORS,
                           title="Plant activity at time of fire"), use_container_width=True)

# ---------------- Equipment risk ----------------
with tabs[5]:
    er = equipment_risk(f, inc["incident_date"].max())
    if er.empty:
        st.info("No tagged equipment in the selection.")
    else:
        top = er.head(15)
        st.plotly_chart(px.bar(top[::-1], x="risk_score", y="primary_tag_base", color="risk_band", orientation="h",
                               color_discrete_map=SEV_COLORS, hover_data=["area", "fires", "last_fire"],
                               title="Equipment risk ranking (fires × severity × injuries × recency)"),
                        use_container_width=True)
        st.dataframe(er[["risk_rank", "primary_tag_base", "area", "equipment_type", "fires", "avg_severity",
                         "injuries", "first_fire", "last_fire", "days_since_last", "risk_score", "risk_band"]],
                     use_container_width=True, hide_index=True)
    et = f.groupby("equipment_type").agg(fires=("incident_id", "count"),
                                         high=("severity_level", lambda s: (s == "High").sum())).reset_index()
    st.plotly_chart(px.treemap(et, path=["equipment_type"], values="fires", color="high",
                               color_continuous_scale="Reds", title="Fires by equipment type (colour = high severity)"),
                    use_container_width=True)

# ---------------- ML ----------------
with tabs[6]:
    if "area_risk" in ml:
        ar = ml["area_risk"]
        st.plotly_chart(px.bar(ar.sort_values("risk_score_30d"), x="risk_score_30d", y="area", color="risk_band",
                               orientation="h", color_discrete_map=SEV_COLORS,
                               hover_data=["prob_ml_30d", "prob_hist_30d", "fires_ytd"],
                               title="Probability of ≥1 fire in the next 30 days"), use_container_width=True)
        st.dataframe(ar, use_container_width=True, hide_index=True)
    else:
        st.info("Run the ML notebook to create fire_ml_models/area_risk_30d.csv")
    if "metrics" in ml:
        st.subheader("Model performance"); st.dataframe(ml["metrics"], use_container_width=True, hide_index=True)

# ---------------- Reports ----------------
with tabs[7]:
    st.write("Download the filtered data and summary tables.")
    er = equipment_risk(f, inc["incident_date"].max())
    summary = pd.DataFrame({
        "KPI": ["Total fires", "Avg response (min)", "% within target", "% out before crew", "Injury incidents",
                "High severity", "Repeat-equipment fires"],
        "Value": [len(f), round(f.response_minutes.mean(), 2),
                  round((f.response_minutes.dropna() <= RESPONSE_TARGET_MIN).mean() * 100, 1),
                  round(f.out_before_fire_crew.mean() * 100, 1), int(f.injury_reported.sum()),
                  int((f.severity_level == "High").sum()), int(f.is_repeat_equipment.sum())]})
    sheets = {"KPI_Summary": summary,
              "Monthly": f.groupby(["ym", "fire_nature"]).size().unstack(fill_value=0).reset_index(),
              "By_Area": f.groupby("area").size().sort_values(ascending=False).reset_index(name="fires"),
              "By_Cause": f["cause_category"].value_counts().reset_index(),
              "Equipment_Risk": er,
              "Incidents": f.drop(columns=["ym"])}
    if "area_risk" in ml:
        sheets["Area_Risk_30d"] = ml["area_risk"]
    stamp = datetime.now().strftime("%Y%m%d")
    st.download_button("⬇️ Excel report", to_excel(sheets), f"Fire_Dashboard_Report_{stamp}.xlsx",
                       "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet")
    st.download_button("⬇️ Filtered incidents (CSV)", f.to_csv(index=False).encode("utf-8-sig"),
                       f"fire_incidents_filtered_{stamp}.csv", "text/csv")
    if not er.empty:
        st.download_button("⬇️ Equipment risk ranking (CSV)", er.to_csv(index=False).encode("utf-8-sig"),
                           f"equipment_risk_{stamp}.csv", "text/csv")
"""
app_path = DATA_DIR / "fire_dashboard_app.py"
app_path.write_text(APP_CODE, encoding="utf-8")
print("Streamlit app written to", app_path.resolve())
print("Run:  streamlit run", app_path.name)